# MOSAIC Provider & Facility Name Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

## What this detects

Provider and facility name quality across five dimensions, driven by the MOSAIC provider/facility naming procedure:

| Dimension | Key checks |
|---|---|
| **Authoritative sourcing** | NPI column present; NPI registry source documented |
| **NPI validity** | 10-digit format; NULL rates; Luhn check-digit; duplicate NPIs |
| **Name formatting** | UPPERCASE compliance; whitespace/punctuation; placeholder values |
| **Deduplication** | Same NPI with multiple name variants; name cardinality vs. NPI cardinality |
| **Privacy** | Provider directories may contain PII |

## Detection approach

This detector works at **table level** — it first identifies tables that contain provider/facility name or NPI columns, then evaluates them as a group. A table with both a name column and an NPI column gets the full NPI + name integrity evaluation. A table with a name column only gets formatting and dedup checks.

**AI use:** AI confirms whether a name column stores provider/facility names (not patient names), classifies provider type (individual vs. organization), detects placeholder values, and assesses whether names appear concatenated or parsed.

| Cell | What it does |
|---|---|
| 1-2 | Overview and Rule Reference |
| 3 | Config / Widgets |
| 4 | Constants: NPI validation, name patterns, placeholder list, standardization rules |
| 5 | Discovery: identify provider name and NPI columns per table |
| 6 | Profiler: null rates, NPI format validity, uppercase compliance, whitespace, dedup signals |
| 7 | AI Classifier: confirm provider name columns, classify type, detect placeholders |
| 8 | Rule Engine: evaluate all requirements per table |
| 9 | Write Results |
| 10 | Summary: blocking → NPI validity → formatting → dedup → privacy → work queue |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed.


# MOSAIC Provider & Facility Name Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## §Authoritative sourcing
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_NPI` | Provider name column present but no NPI column in the table | Add NPI column; prefer NPI Registry as authoritative source (§Auth) |
| `NPI_REGISTRY_NOT_SOURCED` | NPI present but no registry_source / npi_source / source_system column | Document NPI provenance; note when NPI Registry was last reconciled |
| `NPI_NOT_REQUIRED_DOCUMENTED` | Providers without NPI present; no steward-documented exception signal | Document exception with surrogate key strategy per §Auth |

## §NPI validation
| Rule tag | What it detects | Required action |
|---|---|---|
| `INVALID_NPI_FORMAT` | NPI value is not exactly 10 numeric digits | Correct at source or quarantine; NPI must be 10-digit string |
| `NPI_NULL_ROWS` | NULL NPI where NPI column is present | Investigate; NULL NPI acceptable only for documented non-NPI staff |
| `HIGH_NPI_NULL_RATE` | > threshold % of NPI values are NULL | Escalate; systematic NPI loss is certification blocker |
| `NPI_LUHN_INVALID` | NPI fails CMS Luhn check-digit validation (when enabled) | Correct at source; invalid check digit indicates data entry error |
| `NPI_DUPLICATE` | Same NPI appears in more than one row (within this table) | Investigate; deduplicate; merge name variants per §Dedup |

## §Formatting
| Rule tag | What it detects | Required action |
|---|---|---|
| `NAME_NOT_UPPERCASE` | Provider/facility name not in UPPERCASE (when org preference applies) | Apply UPPER(TRIM(col)) in Silver ETL; document casing choice per product |
| `WHITESPACE_IN_NAME` | Leading/trailing whitespace or internal double spaces in name | Trim and collapse in Silver ETL |
| `PUNCTUATION_ABNORMAL` | Trailing commas, periods at end, or unusual punctuation patterns | Clean in Silver ETL; never sacrifice accuracy for formatting |
| `NAME_PLACEHOLDER` | Placeholder/garbage values: UNKNOWN, TEST, N/A, NULL, TBD, etc. | Quarantine or NULL; do not load invalid names into certified dimensions |
| `CONCATENATED_NAME` | Full name stored as one string when first/last columns are separately available | Separate or document as intentional; parsed form preferred |

## §Deduplication
| Rule tag | What it detects | Required action |
|---|---|---|
| `DUPLICATE_NPI_VARIANTS` | Same NPI with multiple distinct name variants (punctuation/spacing differences) | Merge via master list; apply NPI-keyed deduplication per §Dedup |
| `HIGH_NAME_CARDINALITY` | Name column cardinality significantly exceeds NPI cardinality — suggests name variants | Investigate; normalize to NPI-keyed canonical name |
| `MISSING_MASTER_KEY` | No NPI and no steward-defined surrogate key column detectable | Define and document natural key strategy per §Dedup |

## §Privacy
| Rule tag | What it detects | Required action |
|---|---|---|
| `PII_PROVIDER_RISK` | Provider name/directory column detected; may contain PII | Verify access controls and logging policy per §Privacy |

---

## Enforcement
Publishing certified provider dimensions with unmapped duplicates or unvalidated NPIs is a **certification blocker** until remediated.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set, Optional
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "provider_name_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",
    "layer":        _w("layer",        "Unknown"),
    "sample_size":  int(_w("sample_size",  10_000)),
    "seed":         int(_w("seed",         42)),

    # % NULL NPI before HIGH_NPI_NULL_RATE fires
    "npi_null_threshold":    float(_w("npi_null_threshold",    "5.0")),
    # % names not UPPERCASE before NAME_NOT_UPPERCASE fires
    "case_threshold":        float(_w("case_threshold",        "2.0")),
    # % names with whitespace issues before WHITESPACE_IN_NAME fires
    "whitespace_threshold":  float(_w("whitespace_threshold",  "1.0")),
    # Name cardinality / NPI cardinality ratio before HIGH_NAME_CARDINALITY fires
    "name_npi_ratio":        float(_w("name_npi_ratio",        "1.5")),

    # Whether to run Luhn check-digit validation on NPI values
    # (more accurate but requires a UDF; disable for basic profiling)
    "check_luhn":   _w("check_luhn",   "false").strip().lower() == "true",

    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "provider_name_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

print(f"Run          : {RUN_ID}")
print(f"Scope        : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer        : {CFG['layer']}")
print(f"Luhn check   : {CFG['check_luhn']}")
print(f"AI           : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off'}")
print(f"Results      : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------

TAGS = {
    # §Authoritative sourcing
    "MISSING_NPI":                    "§Authoritative-sourcing",
    "NPI_REGISTRY_NOT_SOURCED":       "§Authoritative-sourcing",
    "NPI_NOT_REQUIRED_DOCUMENTED":    "§Authoritative-sourcing",
    # §NPI validation
    "INVALID_NPI_FORMAT":             "§NPI-validation",
    "NPI_NULL_ROWS":                  "§NPI-validation",
    "HIGH_NPI_NULL_RATE":             "§NPI-validation",
    "NPI_LUHN_INVALID":               "§NPI-validation",
    "NPI_DUPLICATE":                  "§NPI-validation",
    # §Formatting
    "NAME_NOT_UPPERCASE":             "§Formatting",
    "WHITESPACE_IN_NAME":             "§Formatting",
    "PUNCTUATION_ABNORMAL":           "§Formatting",
    "NAME_PLACEHOLDER":               "§Formatting",
    "CONCATENATED_NAME":              "§Formatting",
    # §Deduplication
    "DUPLICATE_NPI_VARIANTS":         "§Deduplication",
    "HIGH_NAME_CARDINALITY":          "§Deduplication",
    "MISSING_MASTER_KEY":             "§Deduplication",
    # §Privacy
    "PII_PROVIDER_RISK":              "§Privacy",
}

SEVERITY: Dict[str, int] = {
    "INVALID_NPI_FORMAT":             10,
    "NPI_LUHN_INVALID":               10,
    "HIGH_NPI_NULL_RATE":              9,
    "NAME_PLACEHOLDER":                9,
    "MISSING_NPI":                     8,
    "NPI_NULL_ROWS":                   8,
    "DUPLICATE_NPI_VARIANTS":          8,
    "NPI_DUPLICATE":                   7,
    "NAME_NOT_UPPERCASE":              6,
    "WHITESPACE_IN_NAME":              6,
    "HIGH_NAME_CARDINALITY":           6,
    "PUNCTUATION_ABNORMAL":            5,
    "MISSING_MASTER_KEY":              5,
    "NPI_REGISTRY_NOT_SOURCED":        4,
    "NPI_NOT_REQUIRED_DOCUMENTED":     4,
    "CONCATENATED_NAME":               3,
    "PII_PROVIDER_RISK":               3,
}

# ── Column name patterns ──────────────────────────────────────────────────────
RE_PROVIDER_NAME = re.compile(
    r"\b(provider_name|physician_name|doctor_name|clinician_name|"
    r"practitioner_name|rendering_provider|billing_provider|attending_name|"
    r"referring_name|ordering_provider|supervising_provider|"
    r"facility_name|hospital_name|practice_name|clinic_name|"
    r"organization_name|org_name|group_name|payer_name|"
    r"rendering_name|billing_name|attending_physician|"
    r"provider_full_name|provider_display_name|legal_name|"
    r"dba_name|doing_business_as)\b",
    re.I
)
RE_FIRST_LAST = re.compile(
    r"\b(first_name|last_name|given_name|family_name|middle_name|"
    r"provider_first|provider_last|provider_first_name|provider_last_name|"
    r"physician_first|physician_last|npi_first|npi_last|"
    r"individual_first|individual_last)\b",
    re.I
)
RE_NPI = re.compile(
    r"\b(npi|npi_number|npi_id|provider_npi|billing_npi|rendering_npi|"
    r"attending_npi|referring_npi|ordering_npi|supervising_npi|"
    r"individual_npi|organizational_npi|type1_npi|type2_npi|"
    r"npi_1|npi_2|npi_code|npi_txt)\b",
    re.I
)
RE_NPI_SOURCE = re.compile(
    r"\b(npi_source|registry_source|npi_registry|source_system|"
    r"npi_last_updated|npi_verified|npi_status|npi_deactivation|"
    r"enumeration_date|last_update_date)\b",
    re.I
)
RE_SURROGATE_KEY = re.compile(
    r"\b(provider_id|facility_id|physician_id|clinician_id|"
    r"practitioner_id|rendering_id|billing_id|org_id|"
    r"tin|tax_id|ein|license_number|dea_number|upin|"
    r"natural_key|master_id|golden_id|source_provider_id)\b",
    re.I
)

# Exclude patient-facing name columns from provider name detection
RE_PATIENT_NAME = re.compile(
    r"\b(patient_name|patient_first|patient_last|member_name|"
    r"member_first|member_last|subscriber_name|insured_name|"
    r"beneficiary_name|enrollee_name|contact_name|emergency_contact)\b",
    re.I
)
RE_STAFF_NON_NPI = re.compile(
    r"\b(staff_name|employee_name|user_name|created_by|modified_by|"
    r"entered_by|updated_by|assigned_to|authorized_by)\b",
    re.I
)

# ── NPI validation ────────────────────────────────────────────────────────────
NPI_REGEX = re.compile(r"^[0-9]{10}$")

def _luhn_valid(npi: str) -> bool:
    # CMS NPI Luhn check: prefix with 80840, then standard Luhn algorithm
    s = "80840" + str(npi)
    total = 0
    for i, ch in enumerate(reversed(s)):
        d = int(ch)
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

# ── Name placeholder values ───────────────────────────────────────────────────
NAME_PLACEHOLDERS = {
    "unknown", "test", "n/a", "na", "null", "none", "tbd", "tba",
    "not available", "not applicable", "to be determined",
    "dummy", "fake", "sample", "example", "placeholder",
    "provider name", "facility name", "hospital name",
    "xxx", "yyy", "zzz", "aaa", "bbb",
    "123", "1234", "12345",
}

STANDARDIZATION_RULES: Dict[str, list] = {

    "MISSING_NPI": [
        {"option_key": "add_npi_column",
         "label":      "Add NPI column sourced from NPI Registry",
         "sql":        "-- Add npi STRING; populate from NPI Registry (CMS-approved feed).",
         "notes":      "Prefer NPI Registry to populate and refresh provider legal names, "
                       "taxonomy, and address attributes when NPI is present (§Auth). "
                       "Document exceptions for staff not requiring NPI."},
    ],

    "NPI_REGISTRY_NOT_SOURCED": [
        {"option_key": "add_source_column",
         "label":      "Add npi_source / registry_source column and document reconciliation cadence",
         "sql":        "-- ADD COLUMN npi_source STRING DEFAULT 'NPI_REGISTRY'",
         "notes":      "Periodic reconciliation against NPI Registry for active providers; "
                       "cadence defined per data product SLA (§Validation). "
                       "Document when NPI was last verified against the registry."},
    ],

    "NPI_NOT_REQUIRED_DOCUMENTED": [
        {"option_key": "document_npi_exception",
         "label":      "Document NPI exception with steward-approved surrogate key strategy",
         "sql":        "-- No transform. Document exception in data dictionary.",
         "notes":      "Document exceptions where NPI is unavailable (e.g. staff not requiring NPI) "
                       "with steward-approved surrogate key strategy (§Auth)."},
    ],

    "INVALID_NPI_FORMAT": [
        {"option_key": "quarantine_invalid_npi",
         "label":      "Quarantine rows with invalid NPI format",
         "sql":        "-- Route rows WHERE NOT (npi RLIKE '^[0-9]{10}$') to quarantine.",
         "notes":      "NPI must be exactly 10 numeric digits (CMS requirement). "
                       "Never sacrifice accuracy for formatting -- correct at source (§Format). "
                       "Certified provider dimensions with unvalidated NPIs are certification blockers."},
        {"option_key": "null_invalid_npi",
         "label":      "Set invalid NPI to NULL in Silver",
         "sql":        "-- CASE WHEN npi RLIKE '^[0-9]{10}$' THEN npi ELSE NULL END AS npi",
         "notes":      "Use when quarantine is not available; flag for source correction."},
    ],

    "NPI_NULL_ROWS": [
        {"option_key": "investigate_null_npi",
         "label":      "Investigate NULL NPI rows; document or quarantine",
         "sql":        "-- SELECT * FROM tbl WHERE npi IS NULL",
         "notes":      "NULL NPI acceptable only for documented non-NPI staff (§Auth). "
                       "Reject or flag rows with missing NPI where NPI is required by domain rules (§Valid)."},
    ],

    "HIGH_NPI_NULL_RATE": [
        {"option_key": "escalate_npi_null",
         "label":      "Escalate systematic NPI nulls; block certification",
         "sql":        "-- No transform. Escalate to steward. Block Gold promotion.",
         "notes":      "Publishing certified provider dimensions with unvalidated NPIs "
                       "is a certification blocker (§Enforcement)."},
    ],

    "NPI_LUHN_INVALID": [
        {"option_key": "correct_at_source",
         "label":      "Correct invalid NPI at source; quarantine until corrected",
         "sql":        "-- Route rows where Luhn check fails to quarantine.",
         "notes":      "CMS Luhn check-digit validation catches transposed digits and data entry errors. "
                       "Never sacrifice accuracy for formatting -- correct at source (§Format)."},
    ],

    "NPI_DUPLICATE": [
        {"option_key": "deduplicate_by_npi",
         "label":      "Deduplicate rows with same NPI; keep canonical name from NPI Registry",
         "sql":        "-- ROW_NUMBER() OVER (PARTITION BY npi ORDER BY npi_last_updated DESC) = 1",
         "notes":      "Build and maintain master list keyed on NPI (§Dedup). "
                       "Merge duplicate rows differing only by punctuation, spacing, or alias. "
                       "Publishing certified provider dimensions with unmapped duplicates "
                       "is a certification blocker (§Enforcement)."},
    ],

    "NAME_NOT_UPPERCASE": [
        {"option_key": "apply_uppercase",
         "label":      "Apply UPPER(TRIM(col)) in Silver ETL",
         "sql":        "-- SILVER: UPPER(TRIM(provider_name)) AS provider_name",
         "notes":      "Apply organization-approved case rules; default UPPERCASE where "
                       "organizational preference is documented (§Format). "
                       "Never sacrifice accuracy for formatting."},
        {"option_key": "document_case_exception",
         "label":      "Document casing exception in data dictionary",
         "sql":        "-- No transform. Document case rule in data dictionary.",
         "notes":      "If mixed case is intentional (legal name preservation), document per product."},
    ],

    "WHITESPACE_IN_NAME": [
        {"option_key": "trim_and_collapse",
         "label":      "Trim and collapse internal whitespace in Silver ETL",
         "sql":        "-- SILVER: TRIM(REGEXP_REPLACE(provider_name, '\\s+', ' ')) AS provider_name",
         "notes":      "Apply after trim, before case normalization."},
    ],

    "PUNCTUATION_ABNORMAL": [
        {"option_key": "clean_punctuation",
         "label":      "Remove trailing punctuation in Silver ETL",
         "sql":        "-- SILVER: TRIM(REGEXP_REPLACE(provider_name, '[.,;]+$', '')) AS provider_name",
         "notes":      "Never sacrifice accuracy for formatting (§Format). "
                       "Only remove clearly erroneous punctuation; preserve legal punctuation (e.g. Jr., III)."},
    ],

    "NAME_PLACEHOLDER": [
        {"option_key": "quarantine_placeholder",
         "label":      "Quarantine rows with placeholder provider names",
         "sql":        "-- Route rows WHERE LOWER(TRIM(provider_name)) IN (<placeholder_list>) to quarantine.",
         "notes":      "Invalid names must be corrected at source or quarantined (§Format). "
                       "Do not load obvious placeholders into certified provider dimensions."},
    ],

    "CONCATENATED_NAME": [
        {"option_key": "document_or_separate",
         "label":      "Document concatenated name as intentional or separate into first/last",
         "sql":        "-- If separating: SPLIT_PART(full_name, ' ', 1) AS first_name, etc.",
         "notes":      "Parsed form preferred when source provides separate fields (§Format). "
                       "Document in data dictionary if concatenated form is intentional."},
    ],

    "DUPLICATE_NPI_VARIANTS": [
        {"option_key": "merge_via_master_list",
         "label":      "Merge name variants via NPI-keyed master list",
         "sql":        "-- JOIN provider_master ON npi = provider_master.npi "
                       "→ emit provider_master.canonical_name",
         "notes":      "Build and maintain master list keyed on NPI (§Dedup). "
                       "Merge duplicate rows differing only by punctuation, spacing, or alias."},
    ],

    "HIGH_NAME_CARDINALITY": [
        {"option_key": "normalize_via_npi",
         "label":      "Normalize name column via NPI-keyed canonical name",
         "sql":        "-- JOIN npi_registry ON npi = npi_registry.npi "
                       "→ emit npi_registry.provider_last_name, npi_registry.provider_first_name",
         "notes":      "High name cardinality relative to NPI count suggests name variants. "
                       "Use NPI Registry as authoritative name source (§Auth). "
                       "Insurance/facility names follow same master-list pattern (§Dedup)."},
    ],

    "MISSING_MASTER_KEY": [
        {"option_key": "define_surrogate_key",
         "label":      "Define steward-approved surrogate key strategy",
         "sql":        "-- Document natural key: e.g. TIN + last_name + first_name for non-NPI providers.",
         "notes":      "Insurance and facility names follow master-list pattern with "
                       "steward-defined natural keys (§Dedup). "
                       "Document exception in data dictionary."},
    ],

    "PII_PROVIDER_RISK": [
        {"option_key": "verify_access_controls",
         "label":      "Verify access controls and logging policy for provider directory",
         "sql":        "-- No transform. Verify column-level security in Unity Catalog.",
         "notes":      "Provider directories may contain PII -- apply security & privacy policies "
                       "for access and logging (§Privacy)."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Name placeholders: {len(NAME_PLACEHOLDERS)}")


In [0]:
# ---------------------------------------------------------------------------
# DISCOVERY:
# Per table: identify provider name columns, NPI columns, first/last name
# columns, source/registry columns, and surrogate key columns.
# Builds: table_groups -- per table, all provider-relevant column inventory.
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch}")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)
table_all_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM `{_esc(cat)}`.information_schema.columns
    WHERE table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    table_all_cols[r.table_name].append((r.column_name, r.data_type.upper()))


def _inventory_table(tbl: str, cols: List[Tuple[str, str]]) -> Optional[dict]:
    col_lower = {c.lower(): (c, dt) for c, dt in cols}

    # Identify column groups
    name_cols   = [c for c, _ in cols if RE_PROVIDER_NAME.search(c)
                   and not RE_PATIENT_NAME.search(c)
                   and not RE_STAFF_NON_NPI.search(c)]
    first_last  = [c for c, _ in cols if RE_FIRST_LAST.search(c)
                   and not RE_PATIENT_NAME.search(c)]
    npi_cols    = [c for c, _ in cols if RE_NPI.search(c)]
    src_cols    = [c for c, _ in cols if RE_NPI_SOURCE.search(c)]
    surr_cols   = [c for c, _ in cols if RE_SURROGATE_KEY.search(c)
                   and not RE_NPI.search(c)]

    # A table qualifies for evaluation if it has ANY provider name or NPI column
    if not name_cols and not first_last and not npi_cols:
        return None

    return {
        "name_cols":      name_cols,
        "first_last_cols":first_last,
        "npi_cols":       npi_cols,
        "src_cols":       src_cols,
        "surr_cols":      surr_cols,
        "all_cols":       [c for c, _ in cols],
        "col_dtypes":     {c: dt for c, dt in cols},
        # Profiler stats (filled in Cell 4)
        "total":          0,
        "col_stats":      {},   # col -> {null_count, upper_count, ws_count, punct_count, ...}
        "npi_stats":      {},   # npi_col -> {null_count, invalid_count, luhn_count, dup_count, distinct}
        "name_distinct":  {},   # name_col -> distinct_count
        "npi_distinct":   {},   # npi_col -> distinct_count
        "npi_name_map":   {},   # npi -> set of name variants
        # AI results (filled in Cell 5)
        "is_provider_table":  True,
        "provider_type":      "unknown",  # individual | organization | mixed | unknown
        "has_concatenated":   False,
        "placeholder_cols":   {},  # col -> [placeholder_vals]
        "ai_confidence":      "low",
    }


table_groups: Dict[str, dict] = {}
for tbl in tables:
    cols = table_all_cols.get(tbl, [])
    inv  = _inventory_table(tbl, cols)
    if inv:
        table_groups[tbl] = inv

print(f"Scope   : {cat}.{sch}")
print(f"Tables  : {len(tables)}")
print(f"Provider tables: {len(table_groups)}")
for tbl, inv in table_groups.items():
    print(f"  {tbl}")
    print(f"    name_cols  : {inv['name_cols']}")
    print(f"    npi_cols   : {inv['npi_cols']}")
    print(f"    first/last : {inv['first_last_cols']}")
    print(f"    src_cols   : {inv['src_cols']}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER -- one SQL per table for ALL provider column stats.
# Per name column: null_count, upper_count, ws_count, punct_count,
#                  placeholder_count, distinct_count.
# Per NPI column: null_count, invalid_count, dup_count, distinct_count,
#                 luhn_invalid_count (if enabled).
# NPI-to-name variant mapping: collected via GROUP BY for dedup signal.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(fq)
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def profile_table(tbl: str, inv: dict) -> None:
    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    ph_sql = ", ".join(f"'{p}'" for p in NAME_PLACEHOLDERS)

    for col in inv["name_cols"] + inv["first_last_cols"]:
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        v    = f"TRIM({cs})"
        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            f"APPROX_COUNT_DISTINCT({cs}) AS `__acd__{safe}`",
            # UPPERCASE check: has alpha and equals UPPER version
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} = UPPER({v}) AND {v} RLIKE '[A-Za-z]' THEN 1 END) AS `__upper__{safe}`",
            # has alpha (denominator for case check)
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND {v} RLIKE '[A-Za-z]' THEN 1 END) AS `__alpha__{safe}`",
            # whitespace issues
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND "
            f"  ({cs} != TRIM({cs}) OR {v} RLIKE '  ') THEN 1 END) AS `__ws__{safe}`",
            # trailing punctuation
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) RLIKE '[.,;]+$' "
            f"  THEN 1 END) AS `__punct__{safe}`",
            # placeholder values
            f"COUNT(CASE WHEN LOWER(TRIM({cs})) IN ({ph_sql}) "
            f"  THEN 1 END) AS `__ph__{safe}`",
        ]

    for col in inv["npi_cols"]:
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        v    = f"TRIM({cs})"
        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__npi_null__{safe}`",
            f"APPROX_COUNT_DISTINCT({cs}) AS `__npi_acd__{safe}`",
            # invalid format: not exactly 10 digits
            f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
            f"  AND NOT (TRIM({cs}) RLIKE '^[0-9]{{10}}$') "
            f"  THEN 1 END) AS `__npi_inv__{safe}`",
            # duplicate NPI: rows that share an NPI with another row
            # (approx: total - distinct)
            f"COUNT(*) - APPROX_COUNT_DISTINCT({cs}) AS `__npi_dup__{safe}`",
        ]

    try:
        row   = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
        total = row.get("__total__", 0) or 0
        inv["total"] = total

        for col in inv["name_cols"] + inv["first_last_cols"]:
            safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
            inv["col_stats"][col] = {
                "null_count":   int(row.get(f"__null__{safe}",  0) or 0),
                "acd":          int(row.get(f"__acd__{safe}",   0) or 0),
                "upper_count":  int(row.get(f"__upper__{safe}", 0) or 0),
                "alpha_count":  int(row.get(f"__alpha__{safe}", 0) or 0),
                "ws_count":     int(row.get(f"__ws__{safe}",    0) or 0),
                "punct_count":  int(row.get(f"__punct__{safe}", 0) or 0),
                "ph_count":     int(row.get(f"__ph__{safe}",    0) or 0),
            }
            inv["name_distinct"][col] = int(row.get(f"__acd__{safe}", 0) or 0)

        for col in inv["npi_cols"]:
            safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
            inv["npi_stats"][col] = {
                "null_count":   int(row.get(f"__npi_null__{safe}", 0) or 0),
                "distinct":     int(row.get(f"__npi_acd__{safe}",  0) or 0),
                "invalid_count":int(row.get(f"__npi_inv__{safe}",  0) or 0),
                "dup_approx":   max(0, int(row.get(f"__npi_dup__{safe}", 0) or 0)),
            }
            inv["npi_distinct"][col] = int(row.get(f"__npi_acd__{safe}", 0) or 0)

    except Exception as e:
        print(f"  [WARN] Profile failed for {tbl}: {e}")

    # NPI-to-name variant mapping (detect DUPLICATE_NPI_VARIANTS)
    # For each (npi_col, name_col) pair, check if any NPI has > 1 distinct name
    if inv["npi_cols"] and inv["name_cols"]:
        npi_col  = inv["npi_cols"][0]
        name_col = inv["name_cols"][0]
        npi_cs   = f"`{_esc(npi_col)}`"
        name_cs  = f"`{_esc(name_col)}`"
        try:
            variant_rows = spark.sql(f"""
                SELECT {npi_cs} AS npi, COUNT(DISTINCT UPPER(TRIM({name_cs}))) AS name_variants
                FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`
                WHERE {npi_cs} IS NOT NULL AND {name_cs} IS NOT NULL
                GROUP BY {npi_cs}
                HAVING COUNT(DISTINCT UPPER(TRIM({name_cs}))) > 1
                LIMIT 20
            """).collect()
            inv["npi_name_variants"] = {r["npi"]: int(r["name_variants"]) for r in variant_rows}
        except Exception:
            inv["npi_name_variants"] = {}
    else:
        inv["npi_name_variants"] = {}

    # Collect sample values for name cols (for AI and findings)
    sample_df = table_samples.get(tbl)
    if sample_df:
        for col in inv["name_cols"][:3]:
            cs = f"`{_esc(col)}`"
            try:
                vals = [
                    str(r["v"]) for r in
                    sample_df.select(F.col(cs).alias("v"))
                    .filter(F.col("v").isNotNull()).limit(8).collect()
                ]
                inv["col_stats"][col]["sample_vals"] = vals
            except Exception:
                inv["col_stats"][col]["sample_vals"] = []
        for col in inv["npi_cols"][:2]:
            cs = f"`{_esc(col)}`"
            try:
                # Sample invalid NPI values specifically
                inv_vals = [
                    str(r["v"]) for r in
                    sample_df.select(F.col(cs).alias("v"))
                    .filter(
                        F.col("v").isNotNull() &
                        (~F.trim(F.col(cs)).rlike(r"^[0-9]{10}$"))
                    ).limit(5).collect()
                ]
                inv["npi_stats"][col]["invalid_samples"] = inv_vals
            except Exception:
                inv["npi_stats"][col]["invalid_samples"] = []


# Run profiler
table_samples: Dict[str, DataFrame] = {}
for tbl, inv in table_groups.items():
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df
    profile_table(tbl, inv)

print("Profiler done.")
for tbl, inv in table_groups.items():
    total = inv.get("total", 0)
    npi_summary = {col: f"null={s.get('null_count',0)},inv={s.get('invalid_count',0)}"
                   for col, s in inv.get("npi_stats", {}).items()}
    name_summary= {col: f"ph={s.get('ph_count',0)},ws={s.get('ws_count',0)}"
                   for col, s in inv.get("col_stats", {}).items()
                   if col in inv.get("name_cols",[])}
    print(f"  {tbl}: {total:,} rows | NPI={npi_summary} | names={name_summary}")


In [0]:
# ---------------------------------------------------------------------------
# AI CLASSIFIER -- per table.
# Confirms provider name columns, classifies provider type,
# detects placeholder patterns and concatenated name storage.
# ---------------------------------------------------------------------------

BATCH_SIZE = 10

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def classify_tables(tbls: List[str]) -> None:
    if not CFG["enable_ai"] or not tbls:
        return

    for chunk in _chunked(tbls):
        payload = json.dumps([{
            "table":         tbl,
            "name_cols":     table_groups[tbl]["name_cols"][:5],
            "first_last":    table_groups[tbl]["first_last_cols"][:4],
            "npi_cols":      table_groups[tbl]["npi_cols"][:3],
            "name_samples":  {
                col: table_groups[tbl]["col_stats"].get(col, {}).get("sample_vals", [])[:6]
                for col in table_groups[tbl]["name_cols"][:2]
            },
            "has_npi":       len(table_groups[tbl]["npi_cols"]) > 0,
            "total_cols":    len(table_groups[tbl]["all_cols"]),
        } for tbl in chunk if tbl in table_groups], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            "For each table determine: "
            "(1) is_provider_table: true if the name columns store provider, physician, or facility names. "
            "False if they store patient names, member names, or user/staff names unrelated to clinical providers. "
            "(2) provider_type: 'individual' (person name: first/last), 'organization' (facility/group name), "
            "'mixed' (both), or 'unknown'. "
            "(3) has_concatenated_name: true if full name appears to be stored as a single concatenated string "
            "when separate first/last name fields would be more appropriate for individual providers. "
            "(4) placeholder_cols: list of column names that appear to have placeholder/test/garbage name values. "
            "(5) confidence: high/medium/low. "
            f"Tables: {payload}. "
            'Return: [{"table":"<t>","is_provider_table":true|false,"provider_type":"<p>",'
            '"has_concatenated_name":false,"placeholder_cols":[],"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                tbl = item.get("table", "")
                if tbl not in table_groups:
                    continue
                table_groups[tbl].update({
                    "is_provider_table":  item.get("is_provider_table", True),
                    "provider_type":      item.get("provider_type", "unknown"),
                    "has_concatenated":   item.get("has_concatenated_name", False),
                    "placeholder_cols":   {col: [] for col in item.get("placeholder_cols", [])},
                    "ai_confidence":      item.get("confidence", "low"),
                })
        except Exception as e:
            print(f"  [WARN] AI classification failed for chunk: {e}")
            for tbl in chunk:
                if tbl in table_groups:
                    table_groups[tbl].update({
                        "is_provider_table": True,
                        "provider_type":     "unknown",
                        "has_concatenated":  False,
                        "placeholder_cols":  {},
                        "ai_confidence":     "low",
                        "ai_error":          str(e),
                    })

    # Remove tables AI says are not provider tables (at high confidence)
    for tbl in list(table_groups.keys()):
        inv = table_groups[tbl]
        if not inv.get("is_provider_table", True) and inv.get("ai_confidence") == "high":
            print(f"  [AI] {tbl} excluded: not a provider table")
            del table_groups[tbl]


classify_tables(list(table_groups.keys()))

total_confirmed = len(table_groups)
print(f"AI classification done. {total_confirmed} confirmed provider table(s).")
for tbl, inv in table_groups.items():
    print(f"  {tbl}: type={inv.get('provider_type','?')} "
          f"concat={inv.get('has_concatenated','?')} "
          f"conf={inv.get('ai_confidence','?')}")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, tag, count, total, samples, action,
             hint=None, confidence="high", std_options=None,
             auto_decided_action=None, provider_type="unknown") -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = auto_decided_action if auto_decided_action is not None else None
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "provider_type":            provider_type,
        "layer":                    CFG["layer"],
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


def _check_provider_table(tbl: str, inv: dict) -> list:
    total      = inv.get("total", 0)
    ptype      = inv.get("provider_type", "unknown")
    confidence = inv.get("ai_confidence", "low")
    name_cols  = inv.get("name_cols", [])
    fl_cols    = inv.get("first_last_cols", [])
    npi_cols   = inv.get("npi_cols", [])
    src_cols   = inv.get("src_cols", [])
    surr_cols  = inv.get("surr_cols", [])
    col_stats  = inv.get("col_stats", {})
    npi_stats  = inv.get("npi_stats", {})
    npi_variants = inv.get("npi_name_variants", {})
    findings   = []

    all_name_cols = name_cols + fl_cols

    # ── §Authoritative sourcing: MISSING_NPI ─────────────────────────────────
    # Provider name column present but no NPI column in the table
    if all_name_cols and not npi_cols and ptype != "organization":
        findings.append(_finding(tbl, "-- table --", "MISSING_NPI",
            0, total, [],
            f"Table has provider name column(s) {all_name_cols[:3]} "
            "but no NPI column detected. "
            "Prefer NPI Registry to populate provider attributes when NPI is present (§Auth). "
            "Document exceptions for staff not requiring NPI.",
            confidence=confidence,
            provider_type=ptype,
        ))

    # §Authoritative sourcing: MISSING_MASTER_KEY
    # No NPI and no surrogate key for organization/facility tables
    if not npi_cols and not surr_cols and ptype == "organization":
        findings.append(_finding(tbl, "-- table --", "MISSING_MASTER_KEY",
            0, total, [],
            "Organization/facility table has no NPI and no steward-defined surrogate key detected. "
            "Insurance and facility names follow master-list pattern with steward-defined natural keys (§Dedup).",
            confidence=confidence,
            provider_type=ptype,
        ))

    # §Authoritative sourcing: NPI_REGISTRY_NOT_SOURCED
    if npi_cols and not src_cols:
        findings.append(_finding(tbl, npi_cols[0], "NPI_REGISTRY_NOT_SOURCED",
            0, total, [],
            f"NPI column '{npi_cols[0]}' present but no registry_source or npi_source column detected. "
            "Document NPI provenance and reconciliation cadence (§Auth, §Validation). "
            "Periodic reconciliation against NPI Registry required per data product SLA.",
            confidence="medium",
            provider_type=ptype,
        ))

    # ── §NPI validation ────────────────────────────────────────────────────────
    for npi_col, stats in npi_stats.items():
        npi_null    = stats.get("null_count", 0)
        npi_invalid = stats.get("invalid_count", 0)
        npi_dup     = stats.get("dup_approx", 0)
        npi_dist    = stats.get("distinct", 0)
        inv_samples = stats.get("invalid_samples", [])
        non_null    = total - npi_null

        # INVALID_NPI_FORMAT
        if npi_invalid > 0:
            findings.append(_finding(tbl, npi_col, "INVALID_NPI_FORMAT",
                npi_invalid, total, inv_samples[:5],
                f"{npi_invalid:,} NPI value(s) are not exactly 10 numeric digits. "
                "NPI must be exactly 10 digits (CMS requirement). "
                "Correct at source or quarantine (§Valid). "
                "Certified provider dimensions with unvalidated NPIs are certification blockers.",
                confidence="high",
                provider_type=ptype,
            ))

        # NPI_NULL_ROWS (unconditional -- any null fires)
        if npi_null > 0:
            findings.append(_finding(tbl, npi_col, "NPI_NULL_ROWS",
                npi_null, total, [],
                f"{npi_null:,} row(s) have NULL NPI. "
                "Reject or flag rows with missing NPI where NPI is required by domain rules (§Valid). "
                "Acceptable only for documented non-NPI staff; document exception.",
                confidence="high",
                provider_type=ptype,
            ))

        # HIGH_NPI_NULL_RATE (rate threshold)
        if total > 0 and npi_null / total * 100 > CFG["npi_null_threshold"]:
            findings.append(_finding(tbl, npi_col, "HIGH_NPI_NULL_RATE",
                npi_null, total, [],
                f"{npi_null/total*100:.1f}% of NPI values are NULL. "
                "Publishing certified provider dimensions with unvalidated NPIs "
                "is a certification blocker (§Enforcement).",
                confidence="high",
                auto_decided_action="escalate_npi_null",
                provider_type=ptype,
            ))

        # NPI_DUPLICATE
        if npi_dup > 0:
            findings.append(_finding(tbl, npi_col, "NPI_DUPLICATE",
                npi_dup, total, [],
                f"Approximately {npi_dup:,} row(s) share an NPI with another row. "
                "Build and maintain master list keyed on NPI (§Dedup). "
                "Merge duplicate rows differing only by punctuation, spacing, or alias. "
                "Unmapped duplicates in certified provider dimensions are certification blockers.",
                confidence="medium",
                provider_type=ptype,
            ))

    # ── §Formatting: name column checks ──────────────────────────────────────
    for col in all_name_cols:
        stats    = col_stats.get(col, {})
        null_c   = stats.get("null_count", 0)
        alpha_c  = stats.get("alpha_count", 0)
        upper_c  = stats.get("upper_count", 0)
        ws_c     = stats.get("ws_count", 0)
        punct_c  = stats.get("punct_count", 0)
        ph_c     = stats.get("ph_count", 0)
        samples  = stats.get("sample_vals", [])
        non_null = total - null_c

        # NAME_PLACEHOLDER (unconditional)
        if ph_c > 0:
            findings.append(_finding(tbl, col, "NAME_PLACEHOLDER",
                ph_c, total, samples[:5],
                f"{ph_c:,} value(s) in '{col}' are placeholder/garbage names. "
                "Invalid names must be corrected at source or quarantined (§Format). "
                "Do not load placeholders into certified provider dimensions.",
                confidence="high",
                provider_type=ptype,
            ))

        # WHITESPACE_IN_NAME
        if ws_c > 0 and total > 0 and ws_c / total * 100 > CFG["whitespace_threshold"]:
            findings.append(_finding(tbl, col, "WHITESPACE_IN_NAME",
                ws_c, total, samples[:3],
                f"{ws_c:,} value(s) in '{col}' have leading/trailing whitespace or double spaces. "
                "Trim and collapse in Silver ETL.",
                confidence="high",
                auto_decided_action="trim_and_collapse",
                provider_type=ptype,
            ))

        # PUNCTUATION_ABNORMAL
        if punct_c > 0:
            findings.append(_finding(tbl, col, "PUNCTUATION_ABNORMAL",
                punct_c, total, samples[:3],
                f"{punct_c:,} value(s) in '{col}' have trailing commas or abnormal punctuation. "
                "Clean in Silver ETL; never sacrifice accuracy for formatting (§Format).",
                confidence="medium",
                provider_type=ptype,
            ))

        # NAME_NOT_UPPERCASE (only for UPPERCASE-expected columns)
        # Fires when alpha values present and case compliance is below threshold
        if alpha_c > 0 and non_null > 0:
            non_upper = alpha_c - upper_c
            if non_upper > 0 and non_upper / alpha_c * 100 > CFG["case_threshold"]:
                findings.append(_finding(tbl, col, "NAME_NOT_UPPERCASE",
                    non_upper, total, samples[:3],
                    f"{non_upper:,} value(s) ({non_upper/alpha_c*100:.1f}% of alpha values) "
                    f"in '{col}' are not UPPERCASE. "
                    "Apply organization-approved case rules; default UPPERCASE where "
                    "organizational preference is documented (§Format).",
                    confidence="medium",
                    provider_type=ptype,
                ))

    # ── §Formatting: CONCATENATED_NAME ────────────────────────────────────────
    if inv.get("has_concatenated") and ptype == "individual":
        concat_cols = [c for c in name_cols if not RE_FIRST_LAST.search(c)]
        for col in concat_cols[:2]:
            findings.append(_finding(tbl, col, "CONCATENATED_NAME",
                0, total, col_stats.get(col, {}).get("sample_vals", [])[:3],
                f"Column '{col}' appears to store a concatenated full name for individual providers. "
                "Parsed first/last name form preferred when source provides separate fields (§Format). "
                "Document in data dictionary if concatenated form is intentional.",
                confidence=confidence,
                provider_type=ptype,
            ))

    # ── §Deduplication: DUPLICATE_NPI_VARIANTS ───────────────────────────────
    if npi_variants:
        variant_examples = list(npi_variants.keys())[:5]
        findings.append(_finding(tbl, "-- npi/name --", "DUPLICATE_NPI_VARIANTS",
            len(npi_variants), total, variant_examples,
            f"{len(npi_variants)} NPI value(s) have multiple distinct name variants. "
            "Build and maintain master list keyed on NPI (§Dedup). "
            "Merge duplicate rows differing only by punctuation, spacing, or alias. "
            "Unmapped duplicates in certified provider dimensions are certification blockers.",
            confidence="high",
            provider_type=ptype,
        ))

    # ── §Deduplication: HIGH_NAME_CARDINALITY ─────────────────────────────────
    if name_cols and npi_cols:
        for name_col in name_cols[:1]:
            name_dist = inv["name_distinct"].get(name_col, 0)
            npi_dist  = inv["npi_distinct"].get(npi_cols[0], 1)
            if npi_dist > 0 and name_dist / npi_dist > CFG["name_npi_ratio"]:
                findings.append(_finding(tbl, name_col, "HIGH_NAME_CARDINALITY",
                    name_dist, total, [],
                    f"Name column '{name_col}' has {name_dist:,} distinct values vs. "
                    f"{npi_dist:,} distinct NPI values (ratio: {name_dist/npi_dist:.1f}x). "
                    "High name cardinality relative to NPI count suggests name variants. "
                    "Normalize to NPI-keyed canonical name from NPI Registry (§Auth, §Dedup).",
                    confidence="medium",
                    provider_type=ptype,
                ))

    # ── §Privacy: PII_PROVIDER_RISK ───────────────────────────────────────────
    if all_name_cols or npi_cols:
        findings.append(_finding(tbl, "-- provider directory --", "PII_PROVIDER_RISK",
            total, total, [],
            "Provider directory table detected. Provider directories may contain PII. "
            "Apply security and privacy policies for access and logging (§Privacy).",
            confidence="high",
            auto_decided_action="verify_access_controls",
            provider_type=ptype,
        ))

    return findings


# ── Main loop ─────────────────────────────────────────────────────────────────
all_findings: List[dict] = []

for tbl, inv in table_groups.items():
    findings = _check_provider_table(tbl, inv)
    all_findings.extend(findings)
    print(f"  ok {tbl}: {len(findings)} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("provider_type",            StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "INVALID_NPI_FORMAT", "NPI_LUHN_INVALID", "HIGH_NPI_NULL_RATE",
    "NAME_PLACEHOLDER", "DUPLICATE_NPI_VARIANTS",
}

if not all_findings:
    print("No provider name findings. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS (certification blockers): {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","provider_type","layer","classification",
            "rule_ref","affected_count","affected_pct",
            "sample_values","recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref","provider_type")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.sum("affected_count").alias("total_affected"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- NPI validity
    npi_df = fdf.filter(F.col("rule_ref") == "§NPI-validation")
    n_npi  = npi_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 3 -- NPI validity findings ({n_npi})")
    print("  Invalid NPIs and unvalidated duplicates are certification blockers")
    print("-" * 70)
    if n_npi:
        display(npi_df.select(
            "table_name","column_name","classification",
            "affected_count","affected_pct","sample_values",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 4 -- Authoritative sourcing
    auth_df = fdf.filter(F.col("rule_ref") == "§Authoritative-sourcing")
    n_auth  = auth_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- Authoritative sourcing gaps ({n_auth})")
    print("  NPI Registry preferred source; surrogate key required for non-NPI entities")
    print("-" * 70)
    if n_auth:
        display(auth_df.select(
            "table_name","column_name","provider_type","classification",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 5 -- Formatting
    fmt_df = fdf.filter(F.col("rule_ref") == "§Formatting")
    n_fmt  = fmt_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- Name formatting findings ({n_fmt})")
    print("-" * 70)
    if n_fmt:
        display(fmt_df.select(
            "table_name","column_name","provider_type","classification",
            "affected_count","affected_pct","sample_values",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 6 -- Deduplication
    dedup_df = fdf.filter(F.col("rule_ref") == "§Deduplication")
    n_dedup  = dedup_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- Deduplication findings ({n_dedup})")
    print("  Unmapped duplicates in certified provider dimensions are certification blockers")
    print("-" * 70)
    if n_dedup:
        display(dedup_df.select(
            "table_name","column_name","provider_type","classification",
            "affected_count","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 7 -- Provider table readiness summary
    print("\n" + "-" * 70)
    print("SECTION 7 -- Provider table readiness summary")
    print("-" * 70)
    summary = []
    for tbl, inv in table_groups.items():
        has_npi      = len(inv.get("npi_cols", [])) > 0
        has_src      = len(inv.get("src_cols", [])) > 0
        has_name     = len(inv.get("name_cols", [])) > 0
        has_surr     = len(inv.get("surr_cols", [])) > 0
        npi_invalid  = sum(s.get("invalid_count", 0) for s in inv.get("npi_stats", {}).values())
        npi_null     = sum(s.get("null_count", 0) for s in inv.get("npi_stats", {}).values())
        variants     = len(inv.get("npi_name_variants", {}))
        ready = has_name and has_npi and npi_invalid == 0 and npi_null == 0 and variants == 0
        summary.append({
            "table_name":      tbl,
            "provider_type":   inv.get("provider_type", "?"),
            "has_name_col":    has_name,
            "has_npi_col":     has_npi,
            "has_registry_src":has_src,
            "has_surrogate_key":has_surr,
            "npi_invalid":     npi_invalid,
            "npi_null":        npi_null,
            "npi_variants":    variants,
            "certified_ready": ready,
        })
    if summary:
        display(spark.createDataFrame(summary).orderBy("certified_ready","table_name"))

    # Section 8 -- Work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 8 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","provider_type","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Provider tables: {len(table_groups)}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  NPI: {n_npi}  |  Format: {n_fmt}  |  Dedup: {n_dedup}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
